# Edge Benchmark on the PC (TensorFlow Lite, CPU only)  (v2, 3 Oct 2026)

Reviewer comment #4 asks for latency, FPS, memory and power on edge hardware with INT8 quantization. Without an edge board, this notebook measures everything that can be measured honestly on the PC:

| What | How | Valid beyond this PC? |
|---|---|---|
| INT8 quantization and its **accuracy cost** | TFLite full-integer INT8 (calibrated on training subjects only), mAP on **every held-out subject** vs FP32 | **Yes** - quantized accuracy does not depend on the hardware |
| Model size and compute | TFLite file size (FP32 / FP16 / INT8), parameters, GFLOPs | **Yes** |
| Latency, FPS, memory | TFLite on the CPU only (no GPU), limited to **1 and 4 threads** like a small edge CPU; pre-processing, inference and post-processing timed separately | Indicative only |
| Power | not measured (Windows has no reliable per-process power reading) | - |

**How to report it:** state that no edge board was available, that INT8 accuracy and model size/compute are hardware-independent, and that CPU timings are on a desktop processor restricted to 1/4 threads and are indicative of, not equal to, edge performance. (The Android notebook adds real on-device numbers.)

**Before running:**
1. Main notebook, section 8a, finished for all four models (ONNX exports in `loso_runs/edge_export`). YOLO models must be exported with the latest main notebook (it now records YOLO's anchors).
2. One-time install in the `edge` environment (Anaconda Prompt):
```
conda activate edge
python -m pip install "onnx2tf==1.20.0" "tf_keras==2.15.1" onnx_graphsurgeon sng4onnx psutil
python -m pip install "onnxsim==0.4.36"
```
(the second line is optional - only a fallback for the YOLO conversion). Kernel: **tfod** (the work runs in the `edge` environment).

**Time:** TFLite conversion ~1-3 min per model and fold the first time (16 pairs); accuracy ~15 min; speed ~10 min. Everything finished is skipped on a re-run.

## 1. Settings

In [8]:
import os, sys, re, json, time, shutil, subprocess, statistics
import numpy as np

# ---- Paths (same as the main LOSO notebook) ----
BASE_PATH    = r'd:\RealTimeObjectDetection'
RUNS_ROOT    = os.path.join(BASE_PATH, 'loso_runs')
FOLDS_DIR    = os.path.join(BASE_PATH, 'loso_folds')
RESULTS_ROOT = os.path.join(BASE_PATH, 'runs', 'loso')
EDGE_EXPORT  = os.path.join(RUNS_ROOT, 'edge_export')      # ONNX exports made by section 8a of the main notebook
SCRIPTS_DIR  = os.path.join(RUNS_ROOT, 'scripts')
TFLITE_DIR   = os.path.join(BASE_PATH, 'tflite_bundle')    # TFLite models + held-out test images (shared by both notebooks)

MODELS = ['ssd', 'cbam_ssd', 'yolov5s', 'cbam_yolov5s']
LABELS = {'ssd': 'SSD MobileNetV2 FPNLite 320', 'cbam_ssd': 'CBAM-SSD (proposed)',
          'yolov5s': 'YOLOv5s 416', 'cbam_yolov5s': 'CBAM-YOLOv5s (proposed)'}
TFLITE_VARIANTS = ['fp32', 'fp16', 'int8', 'int8m']   # int8m = INT8 with the CBAM layers kept in float (CBAM-SSD only)
VARIANT_LABEL = {'fp32': 'FP32', 'fp16': 'FP16', 'int8': 'INT8', 'int8m': 'INT8, CBAM in float'}

def variants_for(model):
    return [v for v in TFLITE_VARIANTS if v != 'int8m' or model == 'cbam_ssd']

EDGE_PYTHON = None    # None = the conda environment called 'edge'; or the full path to its python.exe

def find_python(env_name):
    home = os.path.expanduser('~')
    for root in (os.path.join(home, '.conda'), os.path.join(home, 'miniconda3'), os.path.join(home, 'anaconda3'),
                 os.path.join(home, 'miniforge3'), r'C:\ProgramData\miniforge3', r'C:\ProgramData\miniconda3',
                 r'C:\ProgramData\anaconda3'):
        exe = os.path.join(root, 'envs', env_name, 'python.exe' if os.name == 'nt' else os.path.join('bin', 'python'))
        if os.path.exists(exe):
            return exe
    return None

EDGE_PYTHON = EDGE_PYTHON if (EDGE_PYTHON and os.path.exists(EDGE_PYTHON)) else find_python('edge')
FOLDS = sorted((d for d in os.listdir(FOLDS_DIR) if re.fullmatch(r'fold\d+', d)), key=lambda d: int(d[4:]))

def child_env():
    env = os.environ.copy()
    env.pop('PYTHONPATH', None)
    env['PYTHONIOENCODING'] = 'utf-8'
    env['TF_CPP_MIN_LOG_LEVEL'] = '2'
    return env

def run_logged(cmd, log_path):
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    with open(log_path, 'w', encoding='utf-8', errors='replace') as log:
        return subprocess.run([str(c) for c in cmd], stdout=log, stderr=subprocess.STDOUT, env=child_env()).returncode

def run_live(cmd, log_path, keep=lambda line: True):
    """Run a command and show its output here as it comes (also saved to log_path)."""
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    with open(log_path, 'w', encoding='utf-8', errors='replace') as log:
        p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=child_env(),
                             universal_newlines=True, encoding='utf-8', errors='replace', bufsize=1)
        for line in p.stdout:
            log.write(line)
            if keep(line):
                print(line, end='', flush=True)
        return p.wait()

def tail(path, n=20):
    with open(path, encoding='utf-8', errors='replace') as f:
        return ''.join(f.readlines()[-n:])

def mean_std(v):
    v = [x for x in v if x is not None]
    return (float(np.mean(v)), float(np.std(v))) if v else (None, None)

print('Folds      :', FOLDS)
print('Exports    :', EDGE_EXPORT, '(found)' if os.path.isdir(EDGE_EXPORT) else '(NOT FOUND - run section 8a of the main notebook)')
print('edge python:', EDGE_PYTHON or 'NOT FOUND')


Folds      : ['fold1', 'fold2', 'fold3', 'fold4']
Exports    : d:\RealTimeObjectDetection\loso_runs\edge_export (found)
edge python: C:\ProgramData\miniforge3\envs\edge\python.exe


## 2. Scripts and checks

In [9]:
# ---- Write the helper scripts this notebook runs (into loso_runs/scripts, next to the main notebook's) ----
SCRIPT_FILES = {
    'edge_common.py': r'''"""Edge deployment helpers shared by the PC (ONNX Runtime) and the Jetson (TensorRT): pre-processing, post-processing
(box decoding + NMS, identical to the desktop evaluation), runtimes, memory and power readings.
Only needs numpy and OpenCV (plus tensorrt on the Jetson / onnxruntime on the PC).
Kept compatible with Python 3.6 and old numpy so it also runs on the original Jetson Nano (JetPack 4.6).
Written by LOSO_Pipeline_All_in_One.ipynb - edit the notebook, not this file."""
import os, re, sys, glob, json, time, shutil, ctypes, ctypes.util, threading, subprocess
import numpy as np
import cv2


# ================================================================ pre-processing
def tf_legacy_bilinear(img, h, w):
    """Bilinear resize exactly as the TF Object Detection API does it (tf.compat.v1 resize_images:
    align_corners=False, half_pixel_centers=False). OpenCV samples differently, so it is done here in numpy."""
    H, W = img.shape[:2]
    ys = np.arange(h, dtype=np.float32) * np.float32(H / float(h))
    xs = np.arange(w, dtype=np.float32) * np.float32(W / float(w))
    y0 = np.floor(ys).astype(np.int64); x0 = np.floor(xs).astype(np.int64)
    y1 = np.minimum(y0 + 1, H - 1); x1 = np.minimum(x0 + 1, W - 1)
    dy = (ys - y0).astype(np.float32)[:, None, None]; dx = (xs - x0).astype(np.float32)[None, :, None]
    # gather the 4 neighbours first (still uint8), then interpolate in float32: same result, much less work
    r0, r1 = img[y0], img[y1]
    a, b = r0[:, x0].astype(np.float32), r0[:, x1].astype(np.float32)
    c, d = r1[:, x0].astype(np.float32), r1[:, x1].astype(np.float32)
    top = a + (b - a) * dx
    bot = c + (d - c) * dx
    return top + (bot - top) * dy


def preprocess_ssd(img_rgb, meta):
    """SSD: resize to the model's fixed size like the TF fixed_shape_resizer; scaling to [-1, 1] is inside the model."""
    h, w = meta['size']
    return tf_legacy_bilinear(img_rgb, h, w)[None], None


def letterbox(img, new_shape, color=(114, 114, 114)):
    """YOLOv5 letterbox (auto=False): keep the aspect ratio, pad to new_shape (h, w). Same rounding as YOLOv5."""
    shape = img.shape[:2]
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    new_unpad = (int(round(shape[1] * r)), int(round(shape[0] * r)))
    dw, dh = (new_shape[1] - new_unpad[0]) / 2.0, (new_shape[0] - new_unpad[1]) / 2.0
    if (shape[1], shape[0]) != new_unpad:
        img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)
    return img, r, (dw, dh)


def preprocess_yolo(img_rgb, meta):
    im, r, pad = letterbox(img_rgb, meta['size'])
    x = np.ascontiguousarray(im.transpose(2, 0, 1)[None], dtype=np.float32) / 255.0
    return x, (img_rgb.shape[:2], r, pad)


def preprocess(img_rgb, meta):
    return (preprocess_ssd if meta['type'] == 'ssd' else preprocess_yolo)(img_rgb, meta)


# ================================================================ NMS
def nms(boxes, scores, iou_thresh, max_keep=None):
    """Greedy NMS, highest score first; a box is suppressed if IoU > iou_thresh (as in TensorFlow and torchvision).
    boxes: [n, 4] as (a1, b1, a2, b2) corners in any consistent order. Returns kept indices in score order."""
    if len(scores) == 0:
        return np.zeros((0,), dtype=np.int64)
    a1, b1, a2, b2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    areas = np.maximum(a2 - a1, 0) * np.maximum(b2 - b1, 0)
    order = np.argsort(-scores, kind='mergesort')
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        if max_keep is not None and len(keep) >= max_keep:
            break
        rest = order[1:]
        inter = (np.maximum(np.minimum(a2[i], a2[rest]) - np.maximum(a1[i], a1[rest]), 0) *
                 np.maximum(np.minimum(b2[i], b2[rest]) - np.maximum(b1[i], b1[rest]), 0))
        union = areas[i] + areas[rest] - inter
        iou = np.where(union > 0, inter / np.maximum(union, 1e-12), 0.0)
        order = rest[iou <= iou_thresh]
    return np.array(keep, dtype=np.int64)


# ================================================================ post-processing
def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def decode_ssd_boxes(enc, anchors, scales):
    """TF Object Detection API faster_rcnn box coder (anchors and boxes as ymin, xmin, ymax, xmax)."""
    ya, xa = (anchors[:, 0] + anchors[:, 2]) / 2, (anchors[:, 1] + anchors[:, 3]) / 2
    ha, wa = anchors[:, 2] - anchors[:, 0], anchors[:, 3] - anchors[:, 1]
    ty, tx, th, tw = enc[:, 0] / scales[0], enc[:, 1] / scales[1], enc[:, 2] / scales[2], enc[:, 3] / scales[3]
    w, h = np.exp(tw) * wa, np.exp(th) * ha
    yc, xc = ty * ha + ya, tx * wa + xa
    return np.stack([yc - h / 2, xc - w / 2, yc + h / 2, xc + w / 2], axis=1)


def postprocess_ssd(outputs, meta, anchors, conf):
    """Same steps as the TF SSD post-processing: decode -> score converter -> per-class NMS -> sort -> clip -> top-k.
    Returns (boxes [ymin, xmin, ymax, xmax] normalised, scores, class indices 0..C-1)."""
    enc, logits = outputs['boxes'].reshape(-1, 4), outputs['classes']
    logits = logits.reshape(enc.shape[0], -1)
    nm = meta['nms']
    scores = logits * meta['logit_scale'] if meta['logit_scale'] != 1.0 else logits
    conv = meta['score_converter']
    if conv == 'SIGMOID':
        scores = _sigmoid(scores)
    elif conv == 'SOFTMAX':
        e = np.exp(scores - scores.max(1, keepdims=True))
        scores = e / e.sum(1, keepdims=True)
    scores = scores[:, 1:]                                         # drop the background column
    thresh = max(nm['score_thresh'], conf)
    cand = np.where((scores > thresh).any(1))[0]
    if cand.size == 0:
        return np.zeros((0, 4), np.float32), np.zeros((0,), np.float32), np.zeros((0,), np.int64)
    boxes = decode_ssd_boxes(enc[cand], anchors[cand], meta['box_scales'])
    scores = scores[cand]
    all_b, all_s, all_c = [], [], []
    for c in range(scores.shape[1]):
        idx = np.where(scores[:, c] > thresh)[0]
        if idx.size == 0:
            continue
        keep = idx[nms(boxes[idx], scores[idx, c], nm['iou_thresh'], nm['max_per_class'])]
        all_b.append(boxes[keep]); all_s.append(scores[keep, c]); all_c.append(np.full(keep.size, c, np.int64))
    if not all_b:
        return np.zeros((0, 4), np.float32), np.zeros((0,), np.float32), np.zeros((0,), np.int64)
    b, s, c = np.concatenate(all_b), np.concatenate(all_s), np.concatenate(all_c)
    order = np.argsort(-s, kind='mergesort')
    b, s, c = np.clip(b[order], 0.0, 1.0), s[order], c[order]
    nonzero = ((b[:, 2] - b[:, 0]) > 0) & ((b[:, 3] - b[:, 1]) > 0)
    b, s, c = b[nonzero], s[nonzero], c[nonzero]
    k = nm['max_total']
    return b[:k], s[:k], c[:k]


def postprocess_yolo(outputs, meta, info, conf):
    """YOLOv5 non_max_suppression (multi_label=False, per-class via class offsets), then boxes back to the original
    image and normalised. Returns (boxes [ymin, xmin, ymax, xmax] normalised, scores, class indices)."""
    (h0, w0), r, (dw, dh) = info
    pred = outputs['output'].reshape(-1, outputs['output'].shape[-1])
    conf_t = max(meta['conf'], conf)
    x = pred[pred[:, 4] > conf_t]
    empty = (np.zeros((0, 4), np.float32), np.zeros((0,), np.float32), np.zeros((0,), np.int64))
    if x.shape[0] == 0:
        return empty
    cls_scores = x[:, 5:] * x[:, 4:5]
    j = cls_scores.argmax(1)
    s = cls_scores[np.arange(len(j)), j]
    m = s > conf_t
    x, s, j = x[m], s[m], j[m]
    if x.shape[0] == 0:
        return empty
    xy, wh = x[:, :2], x[:, 2:4]
    boxes = np.concatenate([xy - wh / 2, xy + wh / 2], 1)          # x1, y1, x2, y2 in the letterboxed image
    order = np.argsort(-s, kind='mergesort')[:meta['max_nms']]
    boxes, s, j = boxes[order], s[order], j[order]
    offset = j[:, None].astype(np.float32) * 7680.0                 # YOLOv5 max_wh: per-class NMS in one pass
    keep = nms(boxes + offset, s, meta['iou'], meta['max_det'])
    boxes, s, j = boxes[keep], s[keep], j[keep]
    boxes[:, [0, 2]] = np.clip((boxes[:, [0, 2]] - dw) / r, 0, w0) / w0
    boxes[:, [1, 3]] = np.clip((boxes[:, [1, 3]] - dh) / r, 0, h0) / h0
    return boxes[:, [1, 0, 3, 2]], s, j


def postprocess(outputs, meta, prep_info, conf=0.0, anchors=None):
    if meta['type'] == 'ssd':
        return postprocess_ssd(outputs, meta, anchors, conf)
    return postprocess_yolo(outputs, meta, prep_info, conf)


def decode_yolo_heads(heads, meta):
    """YOLOv5 Detect-layer decoding in numpy, for models cut before the decoding (the TFLite models: keeping
    pixel boxes and confidences out of the network keeps INT8 accurate). heads = the 3 raw convolution outputs,
    NCHW [1, na*(5+nc), ny, nx] or NHWC [1, ny, nx, na*(5+nc)]. Returns [1, N, 5+nc] like the ONNX 'output0'."""
    H, W = meta['size']
    na, no = len(meta['anchors'][0]), 5 + meta['num_classes']
    out = []
    for stride, anchors in zip(meta['strides'], meta['anchors']):
        ny, nx = int(round(H / stride)), int(round(W / stride))
        h = None
        for a in heads:
            if a.shape[1:] == (na * no, ny, nx):
                h = a.reshape(na, no, ny, nx).transpose(0, 2, 3, 1)          # NCHW -> (a, y, x, no)
            elif a.shape[1:] == (ny, nx, na * no):
                h = a.reshape(ny, nx, na, no).transpose(2, 0, 1, 3)          # NHWC -> (a, y, x, no)
            if h is not None:
                break
        if h is None:
            raise ValueError('no head output for stride %s (shapes %s)' % (stride, [a.shape for a in heads]))
        y = _sigmoid(h.astype(np.float32))
        gy, gx = np.meshgrid(np.arange(ny, dtype=np.float32), np.arange(nx, dtype=np.float32), indexing='ij')
        xy = (y[..., 0:2] * 2.0 + np.stack([gx, gy], -1)[None] - 0.5) * stride
        wh = (y[..., 2:4] * 2.0) ** 2 * np.asarray(anchors, np.float32)[:, None, None, :]
        out.append(np.concatenate([xy, wh, y[..., 4:]], -1).reshape(-1, no))
    return np.concatenate(out, 0)[None]


def name_outputs(raw, meta):
    """Map runtime output names to 'boxes' / 'classes' (SSD) or 'output' (YOLO)."""
    if meta['type'] != 'ssd':
        if len(raw) == 1:
            return {'output': list(raw.values())[0]}
        if meta.get('output_name') in raw:
            return {'output': raw[meta['output_name']]}
        return {'output': decode_yolo_heads(list(raw.values()), meta)}   # model cut before the Detect decoding
    out = {}
    for name, arr in raw.items():
        n = name.lower()
        if 'box' in n:
            out['boxes'] = arr
        elif 'class' in n:
            out['classes'] = arr
    if len(out) != 2:   # fall back to shapes: boxes end in 4
        for name, arr in raw.items():
            out['boxes' if arr.shape[-1] == 4 else 'classes'] = arr
    return out


# ================================================================ runtimes
class OnnxRuntimeModel(object):
    """PC: ONNX Runtime on the CPU (threads = 0 lets ONNX Runtime use all cores)."""
    def __init__(self, onnx_path, threads=0):
        import onnxruntime as ort
        so = ort.SessionOptions()
        so.inter_op_num_threads = 1
        if threads:
            so.intra_op_num_threads = threads
        self.sess = ort.InferenceSession(onnx_path, so, providers=['CPUExecutionProvider'])
        self.input_name = self.sess.get_inputs()[0].name
        self.output_names = [o.name for o in self.sess.get_outputs()]
        self.device_memory_bytes = None

    def infer(self, x):
        res = self.sess.run(self.output_names, {self.input_name: x})
        return dict(zip(self.output_names, res))


_INTERPRETER = []


def tflite_interpreter():
    """The TFLite Interpreter class: tflite_runtime or ai_edge_litert if installed, else TensorFlow's."""
    if not _INTERPRETER:
        for mod in ('tflite_runtime.interpreter', 'ai_edge_litert.interpreter'):
            try:
                _INTERPRETER.append(__import__(mod, fromlist=['Interpreter']).Interpreter)
                break
            except ImportError:
                pass
        if not _INTERPRETER:
            import tensorflow as tf
            _INTERPRETER.append(tf.lite.Interpreter)
    return _INTERPRETER[0]


class TFLiteModel(object):
    """TFLite interpreter. Accepts NCHW input for NHWC models (YOLO converted from ONNX) and handles integer
    input/output tensors."""
    def __init__(self, path, threads=0):
        self.it = tflite_interpreter()(model_path=path, num_threads=threads or None)
        self.it.allocate_tensors()
        self.inp = self.it.get_input_details()[0]
        self.outs = self.it.get_output_details()
        self.device_memory_bytes = None

    def infer(self, x):
        shape = tuple(int(v) for v in self.inp['shape'])
        if x.shape != shape and x.ndim == 4 and (x.shape[0], x.shape[2], x.shape[3], x.shape[1]) == shape:
            x = x.transpose(0, 2, 3, 1)                     # NCHW -> NHWC
        if self.inp['dtype'] != np.float32:
            scale, zero = self.inp['quantization']
            x = np.clip(np.round(x / scale + zero), np.iinfo(self.inp['dtype']).min, np.iinfo(self.inp['dtype']).max)
        self.it.set_tensor(self.inp['index'], np.ascontiguousarray(x, dtype=self.inp['dtype']))
        self.it.invoke()
        out = {}
        for o in self.outs:
            a = self.it.get_tensor(o['index'])
            if o['dtype'] != np.float32:
                scale, zero = o['quantization']
                a = (a.astype(np.float32) - zero) * scale
            out[o['name']] = a
        return out


class _CudaRT(object):
    """Minimal CUDA runtime through ctypes (libcudart ships with JetPack), so no pycuda install is needed."""
    H2D, D2H = 1, 2

    def __init__(self):
        cands = []
        found = ctypes.util.find_library('cudart')
        if found:
            cands.append(found)
        for pat in ('/usr/local/cuda/lib64/libcudart.so*', '/usr/local/cuda-*/lib64/libcudart.so*',
                    '/usr/local/cuda*/targets/aarch64-linux/lib/libcudart.so*',
                    '/usr/lib/aarch64-linux-gnu/libcudart.so*', '/usr/lib/x86_64-linux-gnu/libcudart.so*'):
            cands += sorted(glob.glob(pat))
        self.lib = None
        for c in cands:
            try:
                self.lib = ctypes.CDLL(c)
                break
            except OSError:
                continue
        if self.lib is None:
            raise RuntimeError('libcudart not found (looked in /usr/local/cuda/lib64). Is JetPack installed?')
        L = self.lib
        L.cudaMalloc.argtypes = [ctypes.POINTER(ctypes.c_void_p), ctypes.c_size_t]
        L.cudaFree.argtypes = [ctypes.c_void_p]
        L.cudaMemcpyAsync.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_size_t, ctypes.c_int, ctypes.c_void_p]
        L.cudaStreamCreate.argtypes = [ctypes.POINTER(ctypes.c_void_p)]
        L.cudaStreamSynchronize.argtypes = [ctypes.c_void_p]
        L.cudaStreamDestroy.argtypes = [ctypes.c_void_p]

    def check(self, err, what):
        if err != 0:
            raise RuntimeError('CUDA error %d in %s' % (err, what))

    def malloc(self, nbytes):
        p = ctypes.c_void_p()
        self.check(self.lib.cudaMalloc(ctypes.byref(p), ctypes.c_size_t(nbytes)), 'cudaMalloc')
        return p

    def stream(self):
        s = ctypes.c_void_p()
        self.check(self.lib.cudaStreamCreate(ctypes.byref(s)), 'cudaStreamCreate')
        return s


class TensorRTModel(object):
    """Jetson: a serialized TensorRT engine. Works with TensorRT 8.2 (JetPack 4.6) up to TensorRT 10 (JetPack 6)."""
    def __init__(self, engine_path):
        import tensorrt as trt
        self.trt = trt
        self.cu = _CudaRT()
        self.logger = trt.Logger(trt.Logger.WARNING)
        self.runtime = trt.Runtime(self.logger)
        with open(engine_path, 'rb') as f:
            self.engine = self.runtime.deserialize_cuda_engine(f.read())
        if self.engine is None:
            raise RuntimeError('Could not load the TensorRT engine %s (built with another TensorRT version?)' % engine_path)
        self.context = self.engine.create_execution_context()
        self.stream = self.cu.stream()
        self.new_api = hasattr(self.engine, 'num_io_tensors')
        self.tensors = []
        if self.new_api:
            for i in range(self.engine.num_io_tensors):
                name = self.engine.get_tensor_name(i)
                is_in = self.engine.get_tensor_mode(name) == trt.TensorIOMode.INPUT
                self._add(name, tuple(self.engine.get_tensor_shape(name)), self.engine.get_tensor_dtype(name), is_in)
            for t in self.tensors:
                self.context.set_tensor_address(t['name'], t['dev'].value)
        else:
            for i in range(self.engine.num_bindings):
                self._add(self.engine.get_binding_name(i), tuple(self.engine.get_binding_shape(i)),
                          self.engine.get_binding_dtype(i), self.engine.binding_is_input(i))
            self.bindings = [t['dev'].value for t in self.tensors]
        self.inputs = [t for t in self.tensors if t['is_input']]
        self.outputs = [t for t in self.tensors if not t['is_input']]
        size = None
        for attr in ('device_memory_size_v2', 'device_memory_size'):
            if hasattr(self.engine, attr):
                try:
                    size = int(getattr(self.engine, attr))
                    break
                except Exception:
                    pass
        self.device_memory_bytes = size

    def _add(self, name, shape, dtype, is_input):
        np_dtype = np.dtype(self.trt.nptype(dtype))
        host = np.zeros(shape, dtype=np_dtype)
        dev = self.cu.malloc(host.nbytes)
        self.tensors.append({'name': name, 'shape': shape, 'host': host, 'dev': dev, 'is_input': is_input})

    def infer(self, x):
        L, cu = self.cu.lib, self.cu
        inp = self.inputs[0]
        x = np.ascontiguousarray(x, dtype=inp['host'].dtype).reshape(inp['shape'])
        cu.check(L.cudaMemcpyAsync(inp['dev'], x.ctypes.data_as(ctypes.c_void_p), x.nbytes, cu.H2D, self.stream), 'H2D')
        if self.new_api:
            ok = self.context.execute_async_v3(stream_handle=self.stream.value)
        else:
            ok = self.context.execute_async_v2(bindings=self.bindings, stream_handle=self.stream.value)
        if not ok:
            raise RuntimeError('TensorRT execution failed')
        for o in self.outputs:
            cu.check(L.cudaMemcpyAsync(o['host'].ctypes.data_as(ctypes.c_void_p), o['dev'], o['host'].nbytes,
                                       cu.D2H, self.stream), 'D2H')
        cu.check(L.cudaStreamSynchronize(self.stream), 'sync')
        return dict((o['name'], o['host'].copy()) for o in self.outputs)

    def close(self):
        for t in self.tensors:
            self.cu.lib.cudaFree(t['dev'])
        self.tensors = []


def find_trtexec():
    for p in (shutil.which('trtexec'), '/usr/src/tensorrt/bin/trtexec', '/usr/local/TensorRT/bin/trtexec'):
        if p and os.path.exists(p):
            return p
    return None


def build_engine(onnx_path, engine_path, precision='fp16', log_path=None):
    """Build a TensorRT engine with trtexec (ships with JetPack). Returns trtexec's own GPU timing if reported."""
    import tensorrt as trt
    exe = find_trtexec()
    if exe is None:
        raise RuntimeError('trtexec not found (expected /usr/src/tensorrt/bin/trtexec)')
    major = int(trt.__version__.split('.')[0]); minor = int(trt.__version__.split('.')[1])
    cmd = [exe, '--onnx=' + onnx_path, '--saveEngine=' + engine_path]
    if precision == 'fp16':
        cmd.append('--fp16')
    if (major, minor) < (8, 4):
        cmd.append('--workspace=1024')
    else:
        cmd.append('--memPoolSize=workspace:1024' + ('M' if major >= 10 else ''))
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    if log_path:
        with open(log_path, 'w') as f:
            f.write(' '.join(cmd) + '\n\n' + p.stdout)
    if p.returncode != 0 or not os.path.exists(engine_path):
        raise RuntimeError('trtexec failed for %s:\n%s' % (onnx_path, p.stdout[-2000:]))
    return parse_trtexec_gpu_ms(p.stdout)


def parse_trtexec_gpu_ms(text):
    m = re.search(r'GPU Compute Time: min = ([\d.]+) ms, max = ([\d.]+) ms, mean = ([\d.]+) ms, median = ([\d.]+) ms', text)
    if not m:
        m2 = re.search(r'GPU Compute.*?mean[: =]+([\d.]+) ms', text)
        return {'mean': float(m2.group(1))} if m2 else None
    return {'min': float(m.group(1)), 'max': float(m.group(2)), 'mean': float(m.group(3)), 'median': float(m.group(4))}


# ================================================================ memory and power
def meminfo_mb():
    """System memory in use (MB). On Jetson the GPU shares this RAM, so it includes GPU memory."""
    if not os.path.exists('/proc/meminfo'):
        try:                              # Windows: psutil if installed
            import psutil
            vm = psutil.virtual_memory()
            return (vm.total - vm.available) / 1048576.0
        except ImportError:
            return None
    vals = {}
    with open('/proc/meminfo') as f:
        for line in f:
            k, v = line.split(':', 1)
            vals[k] = int(v.strip().split()[0])
    avail = vals.get('MemAvailable', vals.get('MemFree', 0))
    return (vals['MemTotal'] - avail) / 1024.0


def process_rss_mb():
    cur = peak = None
    if not os.path.exists('/proc/self/status'):
        try:                              # Windows: psutil if installed (peak_wset = peak working set)
            import psutil
            mi = psutil.Process().memory_info()
            peak = getattr(mi, 'peak_wset', None)
            return mi.rss / 1048576.0, (peak / 1048576.0 if peak else None)
        except ImportError:
            return cur, peak
    with open('/proc/self/status') as f:
        for line in f:
            if line.startswith('VmRSS:'):
                cur = int(line.split()[1]) / 1024.0
            elif line.startswith('VmHWM:'):
                peak = int(line.split()[1]) / 1024.0
    return cur, peak


class Tegrastats(object):
    """Board power (total input rail), RAM and GPU load from tegrastats, sampled in the background."""
    POWER_RE = re.compile(r'\b(POM_5V_IN|VDD_IN|VIN_SYS_5V0)\s+(\d+)(?:mW)?/(\d+)')
    GPU_POWER_RE = re.compile(r'\b(POM_5V_GPU|VDD_GPU_SOC|VDD_CPU_GPU_CV|VDD_GPU)\s+(\d+)(?:mW)?/(\d+)')
    RAM_RE = re.compile(r'RAM (\d+)/(\d+)MB')
    GPU_RE = re.compile(r'GR3D_FREQ (\d+)%')

    def __init__(self, interval_ms=100):
        self.exe = shutil.which('tegrastats') or ('/usr/bin/tegrastats' if os.path.exists('/usr/bin/tegrastats') else None)
        self.interval_ms = interval_ms
        self.samples, self.proc, self.thread = [], None, None

    @property
    def available(self):
        return self.exe is not None

    def start(self):
        if not self.available:
            return
        self.samples = []
        self.proc = subprocess.Popen([self.exe, '--interval', str(self.interval_ms)], stdout=subprocess.PIPE,
                                     stderr=subprocess.STDOUT, universal_newlines=True, bufsize=1)
        self.thread = threading.Thread(target=self._read)
        self.thread.daemon = True
        self.thread.start()

    def _read(self):
        for line in self.proc.stdout:
            s = {'t': time.time()}
            m = self.POWER_RE.search(line)
            if m:
                s['power_mw'] = float(m.group(2))
            m = self.GPU_POWER_RE.search(line)
            if m:
                s['gpu_rail_mw'] = float(m.group(2))
            m = self.RAM_RE.search(line)
            if m:
                s['ram_mb'] = float(m.group(1))
            m = self.GPU_RE.search(line)
            if m:
                s['gpu_load'] = float(m.group(1))
            self.samples.append(s)

    def stop(self):
        if self.proc is not None:
            self.proc.terminate()
            try:
                self.proc.wait(timeout=3)
            except Exception:
                self.proc.kill()
            self.proc = None
        return list(self.samples)

    @staticmethod
    def summarize(samples, t0=None, t1=None):
        sel = [s for s in samples if (t0 is None or s['t'] >= t0) and (t1 is None or s['t'] <= t1)]

        def mean_of(key):
            v = [s[key] for s in sel if key in s]
            return float(np.mean(v)) if v else None

        def max_of(key):
            v = [s[key] for s in sel if key in s]
            return float(np.max(v)) if v else None
        pw, gw = mean_of('power_mw'), mean_of('gpu_rail_mw')
        return {'n_samples': len(sel), 'power_w': pw / 1000.0 if pw is not None else None,
                'gpu_rail_w': gw / 1000.0 if gw is not None else None,
                'ram_mb_peak': max_of('ram_mb'), 'gpu_load_pct': mean_of('gpu_load')}


def cpu_name():
    try:
        if os.name == 'nt':
            import winreg
            k = winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE, r'HARDWARE\DESCRIPTION\System\CentralProcessor\0')
            return winreg.QueryValueEx(k, 'ProcessorNameString')[0].strip()
        with open('/proc/cpuinfo') as f:
            for line in f:
                if line.lower().startswith(('model name', 'hardware', 'cpu model')):
                    return line.split(':', 1)[1].strip()
    except Exception:
        pass
    import platform
    return platform.processor() or platform.machine()


def device_info():
    import platform
    info = {'python': sys.version.split()[0], 'numpy': np.__version__, 'opencv': cv2.__version__,
            'cpu': cpu_name(), 'logical_cores': os.cpu_count(), 'os': platform.platform()}
    try:
        import psutil
        info['physical_cores'] = psutil.cpu_count(logical=False)
        info['ram_gb'] = round(psutil.virtual_memory().total / 2 ** 30, 1)
    except ImportError:
        pass
    try:
        import onnxruntime
        info['onnxruntime'] = onnxruntime.__version__
    except Exception:
        pass
    for path, key in (('/proc/device-tree/model', 'device'), ('/etc/nv_tegra_release', 'l4t')):
        try:
            with open(path) as f:
                info[key] = f.read().replace('\x00', '').strip().split('\n')[0]
        except Exception:
            pass
    try:
        import tensorrt as trt
        info['tensorrt'] = trt.__version__
    except Exception:
        pass
    try:
        p = subprocess.run(['nvpmodel', '-q'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
        info['power_mode'] = ' | '.join(l.strip() for l in p.stdout.splitlines() if l.strip())
    except Exception:
        pass
    return info
''',
    'edge_benchmark.py': r'''"""Edge benchmark for the sign-language detectors (reviewer comment #4).

On the Jetson (TensorRT, default):  python3 edge_benchmark.py
On the PC (ONNX Runtime check):     python edge_benchmark.py --backend onnxruntime --accuracy_only
PC CPU benchmark, INT8, 4 threads:  python edge_benchmark.py --backend onnxruntime --variant int8 --threads 4
TFLite models (PC accuracy check):  python edge_benchmark.py --backend tflite --variant int8 --accuracy_only

For every model and fold it measures, on the held-out subject of that fold:
  accuracy  mAP@0.5, precision, recall, F1 at confidence 0.5 (same metric code as the desktop evaluation)
  latency   pre-processing, inference (incl. host<->GPU copies), post-processing (decode + NMS), end to end
  FPS       frames per second end to end (batch 1, frames already in memory, i.e. a live camera stream)
  memory    engine size, increase in system RAM (Jetson GPU memory is part of system RAM), process peak RSS
  power     board input power from tegrastats while running and when idle, energy per frame
Each model/fold runs in its own process so memory readings are not mixed up. Finished runs are skipped
(use --redo to repeat). Results: results/edge_results.md and results/edge_results.json.
Written by LOSO_Pipeline_All_in_One.ipynb - edit the notebook, not this file."""
import os, sys, json, time, argparse, subprocess
HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import numpy as np
import cv2
import edge_common as ec
from loso_common import class_names, CONF_THRESH, list_images, ground_truths_for, ap_per_class, evaluate_at_threshold


def parse():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument('--bundle', default=HERE)
    p.add_argument('--backend', choices=['tensorrt', 'onnxruntime', 'tflite'], default=None)
    p.add_argument('--variant', choices=['fp32', 'fp16', 'int8', 'int8m'], default='fp32',
                   help='ONNX Runtime: fp32 = model.onnx, int8 = model_int8.onnx; TFLite: model_<variant>.tflite')
    p.add_argument('--threads', type=int, default=0, help='CPU threads for ONNX Runtime / TFLite (0 = default)')
    p.add_argument('--models', nargs='*', default=None)
    p.add_argument('--folds', nargs='*', default=None)
    p.add_argument('--precision', choices=['fp16', 'fp32'], default='fp16')
    p.add_argument('--seconds', type=float, default=30.0, help='length of each timed run')
    p.add_argument('--warmup', type=int, default=30)
    p.add_argument('--idle_seconds', type=float, default=15.0)
    p.add_argument('--deploy_conf', type=float, default=CONF_THRESH, help='confidence threshold in the timed run')
    p.add_argument('--accuracy_only', action='store_true')
    p.add_argument('--skip_accuracy', action='store_true')
    p.add_argument('--redo', action='store_true')
    p.add_argument('--out', default=None)
    p.add_argument('--single', nargs=2, default=None, help=argparse.SUPPRESS)
    a = p.parse_args()
    if a.backend is None:
        try:
            import tensorrt  # noqa: F401
            a.backend = 'tensorrt'
        except ImportError:
            a.backend = 'onnxruntime'
    a.out = a.out or os.path.join(a.bundle, {'tensorrt': 'results', 'onnxruntime': 'results_pc',
                                             'tflite': 'results_tflite'}[a.backend])
    return a


def run_tag(args):
    if args.backend == 'tensorrt':
        return args.precision
    return '%s_t%d' % (args.variant, args.threads)


def model_file(args, model, fold):
    d = model_dir(args, model, fold)
    if args.backend == 'tensorrt':
        return engine_path(args, model, fold)
    if args.backend == 'onnxruntime':
        return os.path.join(d, 'model.onnx' if args.variant == 'fp32' else 'model_%s.onnx' % args.variant)
    return os.path.join(d, 'model_%s.tflite' % args.variant)


def model_dir(args, model, fold):
    return os.path.join(args.bundle, 'models', model, fold)


def engine_path(args, model, fold):
    return os.path.join(model_dir(args, model, fold), 'model_%s.engine' % args.precision)


def load_meta(args, model, fold):
    d = model_dir(args, model, fold)
    with open(os.path.join(d, 'meta.json')) as f:
        meta = json.load(f)
    anchors = np.load(os.path.join(d, 'anchors.npy')) if meta['type'] == 'ssd' else None
    desktop = {}
    if os.path.exists(os.path.join(d, 'desktop.json')):
        with open(os.path.join(d, 'desktop.json')) as f:
            desktop = json.load(f)
    return meta, anchors, desktop


def quick_metrics(dets, gts):
    aps = ap_per_class(dets, gts)
    present = [c for c in class_names if not np.isnan(aps[c])]
    r = evaluate_at_threshold(dets, gts, CONF_THRESH)
    return {'mAP': float(np.mean([aps[c] for c in present])),
            'precision': float(np.mean([r[c]['precision'] for c in present])),
            'recall': float(np.mean([r[c]['recall'] for c in present])),
            'f1': float(np.mean([r[c]['f1'] for c in present])),
            'total_fp': int(sum(r[c]['fp'] for c in class_names))}


def detect(rt, meta, anchors, img_rgb, conf):
    x, info = ec.preprocess(img_rgb, meta)
    out = ec.name_outputs(rt.infer(x), meta)
    return ec.postprocess(out, meta, info, conf=conf, anchors=anchors)


def stats(v):
    v = np.asarray(v, dtype=np.float64)
    return {'mean': float(v.mean()), 'median': float(np.median(v)), 'p95': float(np.percentile(v, 95)),
            'std': float(v.std())}


# ---------------------------------------------------------------- one model on one fold (own process)
def run_single(args, model, fold):
    meta, anchors, desktop = load_meta(args, model, fold)
    test_dir = os.path.join(args.bundle, 'data', fold, 'test')
    row = {'model': model, 'fold': fold, 'backend': args.backend, 'tag': run_tag(args),
           'precision': args.precision if args.backend == 'tensorrt' else args.variant, 'threads': args.threads,
           'input': meta['size'], 'desktop': desktop}
    if args.backend == 'tflite':          # load the runtime library first, so the memory below is the model's own
        ec.tflite_interpreter()
    elif args.backend == 'onnxruntime':
        import onnxruntime  # noqa: F401
    names = list_images(test_dir)
    frames = []
    if not args.accuracy_only:                # timed-run frames are loaded before the memory baseline
        frames = [cv2.cvtColor(cv2.imread(os.path.join(test_dir, n)), cv2.COLOR_BGR2RGB) for n in names[:100]]
    rss0 = ec.process_rss_mb()[0]
    mem0 = ec.meminfo_mb()
    if args.backend == 'tensorrt':
        path = engine_path(args, model, fold)
        rt = ec.TensorRTModel(path)
        info_path = path + '.json'
        if os.path.exists(info_path):
            with open(info_path) as f:
                row['trtexec_gpu_ms'] = json.load(f).get('gpu_ms')
    elif args.backend == 'onnxruntime':
        path = model_file(args, model, fold)
        rt = ec.OnnxRuntimeModel(path, threads=args.threads)
    else:
        path = model_file(args, model, fold)
        rt = ec.TFLiteModel(path, threads=args.threads)
    row['model_file_mb'] = os.path.getsize(path) / 1e6
    onnx_path = os.path.join(model_dir(args, model, fold), 'model.onnx')
    row['onnx_mb'] = os.path.getsize(onnx_path) / 1e6 if os.path.exists(onnx_path) else None
    row['engine_device_memory_mb'] = rt.device_memory_bytes / 1e6 if rt.device_memory_bytes else None
    mem_loaded = ec.meminfo_mb()

    if not args.skip_accuracy:
        dets, t0 = [], time.time()
        for n in names:
            img = cv2.cvtColor(cv2.imread(os.path.join(test_dir, n)), cv2.COLOR_BGR2RGB)
            # conf 0.001 = the desktop floor; NMS is greedy by score, so dropping lower scores first changes nothing
            boxes, scores, classes = detect(rt, meta, anchors, img, conf=0.001)
            for b, s, c in zip(boxes, scores, classes):
                if s >= 0.001:                                     # same floor as the desktop evaluation
                    dets.append({'image': n, 'class_name': meta['class_names'][int(c)], 'score': float(s),
                                 'bbox': [float(v) for v in b]})
        row['accuracy'] = quick_metrics(dets, ground_truths_for(test_dir))
        row['n_test_images'] = len(names)
        row['accuracy_pass_s'] = time.time() - t0
        dump_dir = os.path.join(args.out, 'dumps')
        if not os.path.isdir(dump_dir):
            os.makedirs(dump_dir)
        with open(os.path.join(dump_dir, '%s_%s_%s.json' % (model, fold, run_tag(args))), 'w') as f:
            json.dump({'detections': dets, 'n_images': len(names)}, f)

    if not args.accuracy_only:
        for i in range(args.warmup):
            detect(rt, meta, anchors, frames[i % len(frames)], args.deploy_conf)
        tg = ec.Tegrastats(interval_ms=100)
        tg.start()
        time.sleep(0.5)
        pre, inf, post, total, mem_peak = [], [], [], [], mem_loaded
        rss_peak = ec.process_rss_mb()[0] or 0
        t_start = time.time()
        i = 0
        while time.time() - t_start < args.seconds or i < 50:
            img = frames[i % len(frames)]
            t0 = time.perf_counter()
            x, info = ec.preprocess(img, meta)
            t1 = time.perf_counter()
            raw = rt.infer(x)
            t2 = time.perf_counter()
            ec.postprocess(ec.name_outputs(raw, meta), meta, info, conf=args.deploy_conf, anchors=anchors)
            t3 = time.perf_counter()
            pre.append((t1 - t0) * 1e3); inf.append((t2 - t1) * 1e3); post.append((t3 - t2) * 1e3)
            total.append((t3 - t0) * 1e3)
            if i % 20 == 0:
                if mem_peak is not None:
                    mem_peak = max(mem_peak, ec.meminfo_mb() or 0)
                if rss0 is not None:
                    rss_peak = max(rss_peak, ec.process_rss_mb()[0] or 0)
            i += 1
        t_end = time.time()
        samples = tg.stop()
        row['timing_ms'] = {'pre': stats(pre), 'infer': stats(inf), 'post': stats(post), 'total': stats(total)}
        row['frames_timed'] = len(total)
        row['fps'] = 1000.0 / row['timing_ms']['total']['mean']
        row['fps_inference_only'] = 1000.0 / row['timing_ms']['infer']['mean']
        row['deploy_conf'] = args.deploy_conf
        row['power'] = ec.Tegrastats.summarize(samples, t_start, t_end) if samples else None
        if mem0 is not None:
            row['memory_mb'] = {'system_before': mem0, 'after_load': mem_loaded, 'peak_while_running': mem_peak,
                                'increase': mem_peak - mem0}
    row['process_rss_mb'] = ec.process_rss_mb()[1]          # peak memory of this process (model + runtime + data)
    if rss0 is not None and not args.accuracy_only:
        row['model_memory_mb'] = rss_peak - rss0              # memory added by the model while running (runtime library excluded)
    if hasattr(rt, 'close'):
        rt.close()
    return row


# ---------------------------------------------------------------- report
def mean_over(rows, getter):
    v = []
    for r in rows:
        try:
            x = getter(r)
        except (KeyError, TypeError):
            x = None
        if x is not None:
            v.append(x)
    return (float(np.mean(v)), float(np.std(v)), len(v)) if v else (None, None, 0)


def fmt(m, dec=1, pm=True):
    mu, sd, n = m
    if mu is None:
        return '-'
    return ('%.*f ± %.*f' % (dec, mu, dec, sd)) if (pm and n > 1) else ('%.*f' % (dec, mu))


def write_report(args, rows, info, idle):
    models = []
    for r in rows:
        key = (r['model'], r.get('tag', ''))
        if key not in models:
            models.append(key)
    idle_w = idle.get('power_w') if idle else None
    lines = ['# Edge benchmark - %s' % info.get('device', 'unknown device'), '',
             '- Runtime: %s %s, batch 1, frames already in memory%s' % (
                 args.backend, info.get('tensorrt', ''), ' | CPU: %s' % info['cpu'] if info.get('cpu') else ''),
             '- JetPack / L4T: %s' % info.get('l4t', '-'), '- Power mode: %s' % info.get('power_mode', '-'),
             '- Idle board power: %s W' % ('%.2f' % idle_w if idle_w else 'not available'),
             '- Timed runs use confidence %.2f; accuracy uses all detections (as on the PC). Mean ± std over folds.' % args.deploy_conf,
             '']
    hdr = ('| Model | mAP@0.5 (device) | mAP@0.5 (PC) | Precision | Recall | Latency ms (pre / infer / post) | '
           'End-to-end ms | p95 ms | FPS | GPU-only ms (trtexec) | Engine MB | RAM increase MB | Power W | Energy mJ/frame |')
    lines += [hdr, '|' + '---|' * (hdr.count('|') - 1)]
    for key in models:
        rs = [r for r in rows if (r['model'], r.get('tag', '')) == key]
        m = '%s (%s)' % key if key[1] else key[0]
        acc = lambda k: (lambda r: r['accuracy'][k])
        energy = mean_over(rs, lambda r: r['power']['power_w'] / r['fps'] * 1000.0 if r.get('power') and r['power'].get('power_w') else None)
        lines.append('| %s | %s | %s | %s | %s | %s / %s / %s | %s | %s | %s | %s | %s | %s | %s | %s |' % (
            m, fmt(mean_over(rs, acc('mAP')), 3), fmt(mean_over(rs, lambda r: r['desktop']['mAP']), 3),
            fmt(mean_over(rs, acc('precision')), 3), fmt(mean_over(rs, acc('recall')), 3),
            fmt(mean_over(rs, lambda r: r['timing_ms']['pre']['mean']), 1, False),
            fmt(mean_over(rs, lambda r: r['timing_ms']['infer']['mean']), 1, False),
            fmt(mean_over(rs, lambda r: r['timing_ms']['post']['mean']), 1, False),
            fmt(mean_over(rs, lambda r: r['timing_ms']['total']['mean']), 1),
            fmt(mean_over(rs, lambda r: r['timing_ms']['total']['p95']), 1, False),
            fmt(mean_over(rs, lambda r: r['fps']), 1),
            fmt(mean_over(rs, lambda r: r['trtexec_gpu_ms']['mean']), 2, False),
            fmt(mean_over(rs, lambda r: r['model_file_mb']), 1, False),
            fmt(mean_over(rs, lambda r: r['memory_mb']['increase']), 0, False),
            fmt(mean_over(rs, lambda r: r['power']['power_w']), 2), fmt(energy, 1)))
    lines += ['', '## Per fold', '', '| Model | Run | Fold | mAP device | mAP PC | FPS | Power W | RAM increase MB |', '|---|---|---|---|---|---|---|---|']
    for r in rows:
        def safe(fn, dec):
            try:
                v = fn()
                return '%.*f' % (dec, v) if v is not None else '-'
            except (KeyError, TypeError):
                return '-'
        lines.append('| %s | %s | %s | %s | %s | %s | %s | %s |' % (
            r['model'], r.get('tag', ''), r['fold'], safe(lambda: r['accuracy']['mAP'], 3), safe(lambda: r['desktop']['mAP'], 3),
            safe(lambda: r['fps'], 1), safe(lambda: r['power']['power_w'], 2), safe(lambda: r['memory_mb']['increase'], 0)))
    lines += ['', 'mAP (PC) = the desktop evaluation of the same trained model. A device mAP within ~0.01 of it shows the '
              'deployed model is unchanged. Energy per frame = board power while running / FPS.']
    with open(os.path.join(args.out, 'edge_results.md'), 'w') as f:
        f.write('\n'.join(lines) + '\n')
    with open(os.path.join(args.out, 'edge_results.json'), 'w') as f:
        json.dump({'device': info, 'idle': idle, 'settings': vars(args), 'rows': rows}, f, indent=1)
    print('\n'.join(lines))


# ---------------------------------------------------------------- main
def main():
    args = parse()
    if args.single:
        row = run_single(args, *args.single)
        with open(os.path.join(args.out, 'rows', '%s_%s_%s.json' % (args.single[0], args.single[1], run_tag(args))), 'w') as f:
            json.dump(row, f, indent=1)
        return
    for d in (args.out, os.path.join(args.out, 'rows')):
        if not os.path.isdir(d):
            os.makedirs(d)
    models_root = os.path.join(args.bundle, 'models')
    models = args.models or sorted(os.listdir(models_root), key=lambda m: ['ssd', 'cbam_ssd', 'yolov5s', 'cbam_yolov5s'].index(m)
                                   if m in ('ssd', 'cbam_ssd', 'yolov5s', 'cbam_yolov5s') else 99)
    jobs = []
    for m in models:
        for fold in sorted(os.listdir(os.path.join(models_root, m))):
            needed = os.path.join(models_root, m, fold, 'model.onnx') if args.backend == 'tensorrt' else model_file(args, m, fold)
            if (args.folds is None or fold in args.folds) and os.path.exists(needed):
                jobs.append((m, fold))
    info = ec.device_info()
    print('Device: %s | %s | power mode: %s' % (info.get('device', '?'), info.get('l4t', '?'), info.get('power_mode', '?')))
    print('Backend: %s (%s) | %d runs: %s' % (args.backend, run_tag(args), len(jobs), ', '.join('%s/%s' % j for j in jobs)), flush=True)

    # 1) TensorRT engines (built once on this device; several minutes each on a Jetson Nano)
    if args.backend == 'tensorrt':
        for m, fold in jobs:
            path = engine_path(args, m, fold)
            if os.path.exists(path):
                continue
            print('Building TensorRT %s engine for %s/%s ...' % (args.precision, m, fold), flush=True)
            t0 = time.time()
            gpu = ec.build_engine(os.path.join(model_dir(args, m, fold), 'model.onnx'), path, args.precision,
                                  log_path=path + '.log')
            with open(path + '.json', 'w') as f:
                json.dump({'gpu_ms': gpu, 'build_s': time.time() - t0}, f)
            print('   done in %.0f s (trtexec GPU compute %s ms)' % (time.time() - t0, gpu['mean'] if gpu else '?'), flush=True)

    # 2) Idle power
    idle = None
    idle_path = os.path.join(args.out, 'idle.json')
    if not args.accuracy_only:
        if os.path.exists(idle_path) and not args.redo:
            with open(idle_path) as f:
                idle = json.load(f)
        else:
            tg = ec.Tegrastats()
            if tg.available:
                print('Measuring idle power for %.0f s - leave the board alone...' % args.idle_seconds, flush=True)
                tg.start(); time.sleep(args.idle_seconds); idle = ec.Tegrastats.summarize(tg.stop())
                with open(idle_path, 'w') as f:
                    json.dump(idle, f)
                print('   idle: %s W' % ('%.2f' % idle['power_w'] if idle.get('power_w') else 'power not readable'))
            elif args.backend == 'tensorrt':
                print('tegrastats not found - power will not be measured.')

    # 3) One process per model/fold
    rows = []
    for m, fold in jobs:
        row_path = os.path.join(args.out, 'rows', '%s_%s_%s.json' % (m, fold, run_tag(args)))
        if os.path.exists(row_path) and not args.redo:
            print('%s/%s: done before - skipping' % (m, fold))
        else:
            print('%s/%s: running...' % (m, fold), flush=True)
            cmd = [sys.executable, os.path.abspath(__file__), '--single', m, fold, '--bundle', args.bundle,
                   '--backend', args.backend, '--precision', args.precision, '--seconds', str(args.seconds),
                   '--variant', args.variant, '--threads', str(args.threads),
                   '--warmup', str(args.warmup), '--deploy_conf', str(args.deploy_conf), '--out', args.out]
            if args.accuracy_only:
                cmd.append('--accuracy_only')
            if args.skip_accuracy:
                cmd.append('--skip_accuracy')
            rc = subprocess.call(cmd)
            if rc != 0 or not os.path.exists(row_path):
                print('   !!! %s/%s failed (exit code %d) - continuing' % (m, fold, rc))
                continue
        with open(row_path) as f:
            r = json.load(f)
        rows.append(r)
        acc = r.get('accuracy', {})
        print('   mAP %s (PC %s) | %s FPS | %s W' % (
            '%.3f' % acc['mAP'] if acc else '-', '%.3f' % r['desktop']['mAP'] if r['desktop'].get('mAP') else '-',
            '%.1f' % r['fps'] if 'fps' in r else '-',
            '%.2f' % r['power']['power_w'] if r.get('power') and r['power'].get('power_w') else '-'), flush=True)
    if rows:
        print('')
        write_report(args, rows, info, idle)


if __name__ == '__main__':
    main()
''',
    'edge_tflite.py': r'''"""Convert an exported model to TensorFlow Lite: FP32, FP16 and INT8 (full-integer post-training quantization).

  python edge_tflite.py --dir <export dir> --calib_dir <fold's TRAINING images> [--variants fp32 fp16 int8]

SSD / CBAM-SSD: converted from frozen.pb (outputs: raw box encodings + class logits).
YOLOv5s / CBAM-YOLOv5s: model.onnx -> TensorFlow with onnx2tf, cut just before YOLO's Detect decoding, so the
  network outputs the three raw head tensors and the box decoding runs in numpy (edge_common.decode_yolo_heads).
  The decoded output mixes pixel coordinates (0-416) and probabilities (0-1) in one tensor, which a single INT8 scale
  cannot represent - cutting before it is what keeps INT8 accurate.
INT8: weights and activations int8 (TFLITE_BUILTINS_INT8), float input/output, calibrated on images of the fold's
  training subjects only.
INT8M (CBAM-SSD only): the same INT8 model but with the CBAM attention layers kept in float (mixed precision). Each model is checked against the reference outputs (ref.npz) of the original framework.
Writes model_<variant>.tflite and tflite.json into --dir. Runs in the 'edge' environment.
Written by the edge benchmark notebooks - edit the notebook, not this file."""
import os, sys, re, json, random, shutil, argparse
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import numpy as np
import cv2
import edge_common as ec

IMG_EXT = ('.jpg', '.jpeg', '.png')
p = argparse.ArgumentParser()
p.add_argument('--dir', required=True)
p.add_argument('--calib_dir', required=True)
p.add_argument('--variants', nargs='*', default=['fp32', 'fp16', 'int8'])
p.add_argument('--n_calib', type=int, default=200)
p.add_argument('--redo', action='store_true')
args = p.parse_args()
args.dir, args.calib_dir = os.path.abspath(args.dir), os.path.abspath(args.calib_dir)
with open(os.path.join(args.dir, 'meta.json'), encoding='utf-8') as f:
    meta = json.load(f)
import tensorflow as tf

files = sorted(f for f in os.listdir(args.calib_dir) if f.lower().endswith(IMG_EXT))
random.Random(0).shuffle(files)
files = files[:args.n_calib]


def calib_inputs(nhwc):
    for name in files:
        img = cv2.cvtColor(cv2.imread(os.path.join(args.calib_dir, name)), cv2.COLOR_BGR2RGB)
        x = ec.preprocess(img, meta)[0]
        yield [x.transpose(0, 2, 3, 1) if nhwc and x.shape[1] == 3 else x]


# ---------------------------------------------------------------- source graph -> converter factory
if meta['type'] == 'ssd':
    H, W = meta['size']
    inn = [n.split(':')[0] for n in meta['graph_inputs']]
    outn = [n.split(':')[0] for n in meta['graph_outputs']]

    def make_converter():
        return tf.compat.v1.lite.TFLiteConverter.from_frozen_graph(
            os.path.join(args.dir, 'frozen.pb'), inn, outn, {inn[0]: [1, H, W, 3]})
    nhwc = True
else:
    import onnx
    import onnx2tf
    onnx_path = os.path.join(args.dir, 'model.onnx')
    g = onnx.load(onnx_path).graph
    idx = max(int(k) for n in g.node for k in re.findall(r'/model\.(\d+)/', n.name))
    heads = [n.output[0] for n in g.node if n.op_type == 'Conv' and '/model.%d/' % idx in n.name]
    assert len(heads) == len(meta['strides']), 'Detect head convolutions not found: %s' % heads
    sm_dir = os.path.join(args.dir, 'tf_saved_model')
    if args.redo or not os.path.exists(os.path.join(sm_dir, 'saved_model.pb')):
        shutil.rmtree(sm_dir, ignore_errors=True)
        cut_path = os.path.join(args.dir, 'model_heads.onnx')     # the network up to the three head outputs
        onnx.utils.extract_model(onnx_path, cut_path, [g.input[0].name], heads)
        # onnx2tf downloads 20 sample images unless this file is in the working folder: make it from our own images
        sample = os.path.join(args.dir, 'calibration_image_sample_data_20x128x128x3_float32.npy')
        if not os.path.exists(sample):
            imgs = [cv2.resize(cv2.cvtColor(cv2.imread(os.path.join(args.calib_dir, n)), cv2.COLOR_BGR2RGB), (128, 128))
                    for n in (files * 20)[:20]]
            np.save(sample, np.stack(imgs).astype(np.float32) / 255.0)
        cwd = os.getcwd()
        os.chdir(args.dir)
        try:
            try:
                onnx2tf.convert(input_onnx_file_path=cut_path, output_folder_path=sm_dir, output_signaturedefs=True,
                                non_verbose=True, not_use_onnxsim=True)
            except BaseException:          # onnx2tf exits on some graphs; retry with its ONNX simplifier
                print('onnx2tf: first attempt failed - retrying with graph simplification', flush=True)
                shutil.rmtree(sm_dir, ignore_errors=True)
                onnx2tf.convert(input_onnx_file_path=cut_path, output_folder_path=sm_dir, output_signaturedefs=True,
                                non_verbose=True, not_use_onnxsim=False)
        finally:
            os.chdir(cwd)
        if not os.path.exists(os.path.join(sm_dir, 'saved_model.pb')):
            sys.exit('onnx2tf could not convert %s' % cut_path)
        for f_ in os.listdir(sm_dir):                       # onnx2tf's own .tflite files are not used
            if f_.endswith('.tflite'):
                os.remove(os.path.join(sm_dir, f_))

    def make_converter():
        return tf.lite.TFLiteConverter.from_saved_model(sm_dir)
    nhwc = True

# ---------------------------------------------------------------- convert
info = {'variants': {}, 'calibration_images': len(files), 'calib_dir': args.calib_dir}
info_path = os.path.join(args.dir, 'tflite.json')
if os.path.exists(info_path) and not args.redo:
    with open(info_path, encoding='utf-8') as f:
        info = json.load(f)
if meta.get('kind') != 'cbam' and 'int8m' in args.variants:      # mixed precision only applies to CBAM-SSD
    args.variants = [v for v in args.variants if v != 'int8m']
for v in args.variants:
    out = os.path.join(args.dir, 'model_%s.tflite' % v)
    if os.path.exists(out) and v in info['variants'] and not args.redo:
        continue
    if v == 'int8m':
        fp32_path = os.path.join(args.dir, 'model_fp32.tflite')
        if not os.path.exists(fp32_path):
            with open(fp32_path, 'wb') as f:
                f.write(make_converter().convert())
        it = tf.lite.Interpreter(model_path=fp32_path)
        keep_float = [t['name'] for t in it.get_tensor_details() if 'cbam' in t['name'].lower()]
        c = make_converter()
        c.optimizations = [tf.lite.Optimize.DEFAULT]
        c.representative_dataset = lambda: calib_inputs(nhwc)
        dbg = tf.lite.experimental.QuantizationDebugger(
            converter=c, debug_dataset=lambda: calib_inputs(nhwc),
            debug_options=tf.lite.experimental.QuantizationDebugOptions(denylisted_nodes=keep_float))
        with open(out, 'wb') as f:
            f.write(dbg.get_nondebug_quantized_model())
        info['variants'][v] = {'mb': os.path.getsize(out) / 1e6, 'float_tensors': len(keep_float)}
        continue
    c = make_converter()
    if v == 'fp16':
        c.optimizations = [tf.lite.Optimize.DEFAULT]
        c.target_spec.supported_types = [tf.float16]
    elif v == 'int8':
        c.optimizations = [tf.lite.Optimize.DEFAULT]
        c.representative_dataset = lambda: calib_inputs(nhwc)
        c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    with open(out, 'wb') as f:
        f.write(c.convert())
    info['variants'][v] = {'mb': os.path.getsize(out) / 1e6}

# ---------------------------------------------------------------- check against the original framework
ref = np.load(os.path.join(args.dir, 'ref.npz'))
anchors = np.load(os.path.join(args.dir, 'anchors.npy')) if meta['type'] == 'ssd' else None


def dets_of(rt, i):
    x = ref['input_%d' % i]
    o = ec.name_outputs(rt.infer(x), meta)
    pinfo = None
    if meta['type'] != 'ssd':
        pinfo = ((int(ref['orig_shape_%d' % i][0]), int(ref['orig_shape_%d' % i][1])), float(ref['ratio_%d' % i]),
                 (float(ref['pad_%d' % i][0]), float(ref['pad_%d' % i][1])))
    return o, ec.postprocess(o, meta, pinfo, conf=0.0, anchors=anchors)


def agreement(boxes, classes, rb, rs, rc, min_iou):
    hit = tot = 0
    for rank, (bb, ss, cc) in enumerate(zip(rb, rs, rc)):
        if ss < 0.25 and rank >= 10:          # confident reference detections, at least the top 10
            continue
        tot += 1
        same = np.where(classes == cc)[0]
        if same.size:
            y1 = np.maximum(boxes[same, 0], bb[0]); x1 = np.maximum(boxes[same, 1], bb[1])
            y2 = np.minimum(boxes[same, 2], bb[2]); x2 = np.minimum(boxes[same, 3], bb[3])
            inter = np.clip(y2 - y1, 0, None) * np.clip(x2 - x1, 0, None)
            area = lambda q: (q[..., 2] - q[..., 0]) * (q[..., 3] - q[..., 1])
            if np.any(inter / (area(boxes[same]) + area(bb) - inter + 1e-12) > min_iou):
                hit += 1
    return hit, tot


for v in args.variants:
    rt = ec.TFLiteModel(os.path.join(args.dir, 'model_%s.tflite' % v))
    worst, hit, tot = 0.0, 0, 0
    for i in range(len(ref['names'])):
        o, (b, s, c) = dets_of(rt, i)
        if meta['type'] == 'ssd':
            pairs = [(o['boxes'], ref['boxes_%d' % i]), (o['classes'], ref['classes_%d' % i])]
        else:
            pairs = [(o['output'], ref['output_%d' % i])]
        for a, r in pairs:
            rel = float(np.abs(a.reshape(-1) - r.reshape(-1)).max() / (np.abs(r).max() + 1e-9))
            worst = max(worst, rel if np.isfinite(rel) else float('inf'))
        h, t = agreement(b, c, ref['det_boxes_%d' % i], ref['det_scores_%d' % i], ref['det_classes_%d' % i],
                         0.98 if v == 'fp32' else 0.5)
        hit += h; tot += t
    rate = hit / tot if tot else 1.0
    ok = (worst < 1e-3 and rate >= 0.95) if v == 'fp32' else rate >= 0.5
    info['variants'][v].update({'max_relative_output_difference': worst, 'detections_matched': hit,
                                'detections_compared': tot, 'match_rate': rate, 'ok': ok})
    print('%-5s %6.1f MB | vs original: max rel. output diff %.2e | reference detections kept %d/%d (%.0f%%)%s' % (
        v, info['variants'][v]['mb'], worst, hit, tot, 100 * rate, '' if ok else '  !!! CHECK'), flush=True)
with open(info_path, 'w', encoding='utf-8') as f:
    json.dump(info, f, indent=1)
''',
    'edge_pc.py': r'''"""Model complexity (run in the 'edge' environment).

  python edge_pc.py complexity --onnx <model.onnx>
      Parameters and GFLOPs of an ONNX model (multiply-accumulates of Conv / MatMul / Gemm x 2), as JSON.
Written by the edge benchmark notebooks - edit the notebook, not this file."""
import os, sys, json, argparse
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np


def complexity(path):
    import onnx
    from onnx import TensorProto
    m = onnx.shape_inference.infer_shapes(onnx.load(path))
    g = m.graph
    shapes = {}
    for vi in list(g.value_info) + list(g.input) + list(g.output):
        shapes[vi.name] = [d.dim_value for d in vi.type.tensor_type.shape.dim]
    init = {i.name: list(i.dims) for i in g.initializer}
    for n in g.node:                                   # weights stored as Constant nodes
        if n.op_type == 'Constant':
            for a in n.attribute:
                if a.name == 'value':
                    init[n.output[0]] = list(a.t.dims)
    prod = lambda v: int(np.prod(v)) if v else 0
    macs, unknown = 0, 0
    for n in g.node:
        out = shapes.get(n.output[0])
        if n.op_type == 'Conv':
            w = init.get(n.input[1]) or shapes.get(n.input[1])
            if w and out and all(out):
                macs += prod(out) * prod(w[1:])
            else:
                unknown += 1
        elif n.op_type in ('MatMul', 'Gemm'):
            a = shapes.get(n.input[0]) or init.get(n.input[0])
            if a and out and all(out):
                k = a[-2] if (n.op_type == 'Gemm' and any(x.name == 'transA' and x.i for x in n.attribute)) else a[-1]
                macs += prod(out) * k
            else:
                unknown += 1
    params = sum(prod(i.dims) for i in g.initializer if i.data_type in (TensorProto.FLOAT, TensorProto.FLOAT16))
    return {'params': params, 'gmacs': macs / 1e9, 'gflops': 2 * macs / 1e9, 'layers_without_shape': unknown,
            'mb': os.path.getsize(path) / 1e6}


if __name__ == '__main__':
    p = argparse.ArgumentParser()
    sub = p.add_subparsers(dest='cmd')
    c = sub.add_parser('complexity')
    c.add_argument('--onnx', required=True)
    a = p.parse_args()
    if a.cmd == 'complexity':
        print(json.dumps(complexity(a.onnx)))
    else:
        p.print_help()
''',
}
os.makedirs(SCRIPTS_DIR, exist_ok=True)
for name, text in SCRIPT_FILES.items():
    with open(os.path.join(SCRIPTS_DIR, name), 'w', encoding='utf-8') as f:
        f.write(text)
print(f'Wrote {len(SCRIPT_FILES)} scripts to {SCRIPTS_DIR}')

Wrote 4 scripts to d:\RealTimeObjectDetection\loso_runs\scripts


In [10]:
# ---- Check the 'edge' environment and the files from the main notebook ----
EDGE_HOW = ('In Anaconda Prompt:\n'
            '  conda activate edge\n'
            '  python -m pip install "onnx2tf==1.20.0" "tf_keras==2.15.1" onnx_graphsurgeon sng4onnx psutil\n'
            '  python -m pip install "onnxsim==0.4.36"      (optional; only used if a conversion needs it)\n'
            "(The 'edge' environment itself is created in section 8 of the main notebook.)")
ok = True
for f in ('loso_common.py', 'loso_config.json'):
    if not os.path.exists(os.path.join(SCRIPTS_DIR, f)):
        print(f'!!! {f} not found in {SCRIPTS_DIR} - run the main notebook (cell 2c) first.'); ok = False
if not (EDGE_PYTHON and os.path.exists(EDGE_PYTHON)):
    print("!!! 'edge' environment not found. Create it as in section 8 of the main notebook, or set EDGE_PYTHON.")
    ok = False
else:
    r = subprocess.run([EDGE_PYTHON, '-c', 'import tensorflow, onnx, onnxruntime, cv2, onnx2tf, psutil; '
                        'print(tensorflow.__version__, onnx2tf.__version__)'], capture_output=True, text=True, env=child_env())
    if r.returncode != 0:
        print('!!! packages missing in the edge environment:\n   ', (r.stderr.strip().splitlines() or ['?'])[-1])
        print(EDGE_HOW); ok = False
    else:
        print('edge environment OK: TensorFlow %s, onnx2tf %s' % tuple(r.stdout.split()[-2:]))
missing = [f'{m}/{f}' for m in MODELS for f in FOLDS if not os.path.exists(os.path.join(EDGE_EXPORT, m, f, 'validation.json'))]
if missing:
    print(f'!!! {len(missing)} model/fold exports missing (e.g. {missing[0]}): run section 8a of the main notebook first.')
print('Ready.' if ok else 'Fix the points above, then run this cell again.')


edge environment OK: TensorFlow 2.15.1, onnx2tf 1.20.0
Ready.


## 3. Convert to TensorFlow Lite (FP32, FP16, INT8)

SSD / CBAM-SSD are converted from the exported TensorFlow graph. YOLOv5s / CBAM-YOLOv5s are converted from ONNX with onnx2tf, **cut just before YOLO's box decoding** (the decoding then runs in numpy, exactly as YOLOv5 does it): YOLO's final tensor mixes pixel coordinates (0-416) with probabilities (0-1), which one INT8 scale cannot hold, so this cut is what keeps INT8 accurate. Each converted model is checked against the original.

In [11]:
# ---- TFLite conversion (FP32, FP16, INT8) of every model and fold, then the bundle used for testing ----
# INT8 is calibrated on 200 images of the fold's TRAINING subjects; the held-out subject is never seen.
def convert_tflite(model, fold):
    src = os.path.join(EDGE_EXPORT, model, fold)
    if not os.path.exists(os.path.join(src, 'validation.json')):
        return None, 'not exported - run section 8a of the main notebook'
    with open(os.path.join(src, 'meta.json'), encoding='utf-8') as f:
        meta = json.load(f)
    if meta['type'] != 'ssd' and 'anchors' not in meta:
        return None, (f'exported by an older notebook version: delete {src} and run section 8a of the '
                      'latest main notebook again')
    info_path = os.path.join(src, 'tflite.json')
    if os.path.exists(info_path):
        with open(info_path, encoding='utf-8') as f:
            info = json.load(f)
        if all(v in info.get('variants', {}) and 'ok' in info['variants'][v] and
               os.path.exists(os.path.join(src, f'model_{v}.tflite')) for v in variants_for(model)):
            return info, None
    log = os.path.join(src, 'tflite.log')
    rc = run_logged([EDGE_PYTHON, os.path.join(SCRIPTS_DIR, 'edge_tflite.py'), '--dir', src,
                     '--calib_dir', os.path.join(FOLDS_DIR, fold, 'train'), '--variants'] + TFLITE_VARIANTS, log)
    if rc != 0 or not os.path.exists(info_path):
        return None, f'conversion failed (log: {log}):\n{tail(log, 12)}'
    with open(info_path, encoding='utf-8') as f:
        return json.load(f), None

print(f"{'Model':<15}{'Fold':<7}" + ''.join(f'{VARIANT_LABEL[v]:>22}' for v in TFLITE_VARIANTS))
converted = []
for model in MODELS:
    for fold in FOLDS:
        t0 = time.time()
        info, err = convert_tflite(model, fold)
        if err:
            print(f'{model:<15}{fold:<7}  !!! {err}'); continue
        cells = []
        for v in TFLITE_VARIANTS:
            d = info['variants'].get(v)
            cells.append(f"{d['mb']:5.1f} MB {100 * d['match_rate']:3.0f}% kept{'' if d['ok'] else ' !!!'}"
                         if d and 'match_rate' in d else '-')
        print(f'{model:<15}{fold:<7}' + ''.join(f'{c:>22}' for c in cells) + f'   ({time.time() - t0:.0f} s)', flush=True)
        converted.append((model, fold))
print('\n"% kept" = reference detections of the original model that the TFLite model still finds '
      '(FP32: same box and score; FP16/INT8: same class, IoU > 0.5). "!!!" = fewer than half kept on these few '
      'images; judge INT8 by the held-out mAP in the next step.')

# Bundle: TFLite models + meta + Phase B metrics + held-out test images + scripts
for model, fold in converted:
    src, dst = os.path.join(EDGE_EXPORT, model, fold), os.path.join(TFLITE_DIR, 'models', model, fold)
    os.makedirs(dst, exist_ok=True)
    for name in [f'model_{v}.tflite' for v in TFLITE_VARIANTS] + ['meta.json', 'anchors.npy']:
        if os.path.exists(os.path.join(src, name)):
            shutil.copy2(os.path.join(src, name), dst)
    mpath = os.path.join(RESULTS_ROOT, model, f'{fold}_metrics.json')
    if os.path.exists(mpath):
        with open(mpath, encoding='utf-8') as f:
            m = json.load(f)
        with open(os.path.join(dst, 'desktop.json'), 'w', encoding='utf-8') as f:
            json.dump({k: m[k] for k in ('mAP', 'precision', 'recall', 'f1', 'total_fp')}, f, indent=1)
for fold in sorted({f for _, f in converted}):
    dst = os.path.join(TFLITE_DIR, 'data', fold, 'test')
    if not os.path.isdir(dst):
        shutil.copytree(os.path.join(FOLDS_DIR, fold, 'test'), dst)
for name in ('edge_benchmark.py', 'edge_common.py', 'loso_common.py', 'loso_config.json'):
    shutil.copy2(os.path.join(SCRIPTS_DIR, name), TFLITE_DIR)
print(f'\nTFLite bundle: {len(converted)} model/fold pairs in {TFLITE_DIR}')


Model          Fold                     FP32                  FP16                  INT8   INT8, CBAM in float
ssd            fold1       11.3 MB 100% kept      5.7 MB 100% kept      3.3 MB 100% kept                     -   (0 s)
ssd            fold2       11.3 MB 100% kept      5.7 MB 100% kept      3.3 MB 100% kept                     -   (0 s)
ssd            fold3       11.3 MB 100% kept      5.7 MB 100% kept      3.3 MB 100% kept                     -   (0 s)
ssd            fold4       11.3 MB 100% kept      5.7 MB 100% kept      3.3 MB 100% kept                     -   (0 s)
cbam_ssd       fold1       11.4 MB 100% kept      5.8 MB 100% kept      3.3 MB 100% kept      3.6 MB 100% kept   (0 s)
cbam_ssd       fold2       11.4 MB 100% kept      5.8 MB 100% kept      3.3 MB 100% kept      3.6 MB 100% kept   (0 s)
cbam_ssd       fold3       11.4 MB 100% kept      5.8 MB 100% kept      3.3 MB 100% kept      3.6 MB 100% kept   (0 s)
cbam_ssd       fold4       11.4 MB 100% kept      5.8 MB

## 4. Accuracy on every held-out subject (FP32 vs INT8)

In [12]:
# ---- Accuracy of the TFLite models on every held-out subject (FP32 and INT8), on this PC ----
# Same images, same metric code as Phase B. FP16 is not listed: on a CPU it computes in FP32 (same result as FP32);
# its FP16 arithmetic only happens on the phone's GPU.
ACC_VARIANTS = ['fp32', 'int8', 'int8m']        # int8m exists only for CBAM-SSD
bench = os.path.join(TFLITE_DIR, 'edge_benchmark.py')
for v in ACC_VARIANTS:
    print(f'TFLite {v.upper()}: evaluating every held-out subject...', flush=True)
    run_live([EDGE_PYTHON, bench, '--backend', 'tflite', '--variant', v, '--accuracy_only'],
             os.path.join(TFLITE_DIR, f'accuracy_{v}.log'), keep=lambda l: l.rstrip().endswith('running...') or l.startswith('   ') or '!!!' in l)

acc_rows = {}
rows_dir = os.path.join(TFLITE_DIR, 'results_tflite', 'rows')
for fn in os.listdir(rows_dir) if os.path.isdir(rows_dir) else []:
    with open(os.path.join(rows_dir, fn), encoding='utf-8') as f:
        r = json.load(f)
    if r.get('accuracy'):
        acc_rows[(r['model'], r['fold'], r['precision'])] = r
print(f"\n{'Model':<26}{'mAP Phase B':>14}{'TFLite FP32':>14}{'TFLite INT8':>14}{'INT8 - FP32':>13}{'P INT8':>9}{'R INT8':>9}"
      f"{'INT8 CBAM-float':>17}")
ACCURACY = {}
for model in MODELS:
    get = lambda v, k: [acc_rows[(model, f, v)]['accuracy'][k] for f in FOLDS if (model, f, v) in acc_rows]
    pb = [acc_rows[(model, f, 'fp32')]['desktop'].get('mAP') for f in FOLDS if (model, f, 'fp32') in acc_rows]
    f32, i8 = get('fp32', 'mAP'), get('int8', 'mAP')
    if not f32:
        continue
    ACCURACY[model] = {'phaseB': mean_std(pb), 'fp32': mean_std(f32), 'int8': mean_std(i8),
                       'int8m': mean_std(get('int8m', 'mAP')),
                       'p_int8': mean_std(get('int8', 'precision')), 'r_int8': mean_std(get('int8', 'recall')),
                       'n_folds': len(f32)}
    a = ACCURACY[model]
    fmt = lambda t: f'{t[0]:.3f}±{t[1]:.3f}' if t[0] is not None else '-'
    d = (a['int8'][0] - a['fp32'][0]) if a['int8'][0] is not None else None
    print(f"{LABELS[model]:<26}{fmt(a['phaseB']):>14}{fmt(a['fp32']):>14}{fmt(a['int8']):>14}"
          f"{(f'{d:+.3f}' if d is not None else '-'):>13}{a['p_int8'][0] or 0:>9.3f}{a['r_int8'][0] or 0:>9.3f}"
          f"{fmt(a['int8m']):>17}")
print('\nTFLite FP32 should equal Phase B (same model, same images). INT8 - FP32 is the accuracy cost of quantization.')


TFLite FP32: evaluating every held-out subject...
   mAP 0.805 (PC 0.805) | - FPS | - W
   mAP 0.401 (PC 0.401) | - FPS | - W
   mAP 0.928 (PC 0.928) | - FPS | - W
   mAP 0.554 (PC 0.554) | - FPS | - W
   mAP 0.803 (PC 0.803) | - FPS | - W
   mAP 0.431 (PC 0.431) | - FPS | - W
   mAP 0.938 (PC 0.938) | - FPS | - W
   mAP 0.658 (PC 0.658) | - FPS | - W
yolov5s/fold1: running...
   mAP 0.965 (PC 0.965) | - FPS | - W
yolov5s/fold2: running...
   mAP 0.691 (PC 0.691) | - FPS | - W
yolov5s/fold3: running...
   mAP 0.914 (PC 0.914) | - FPS | - W
yolov5s/fold4: running...
   mAP 0.872 (PC 0.872) | - FPS | - W
cbam_yolov5s/fold1: running...
   mAP 0.934 (PC 0.934) | - FPS | - W
cbam_yolov5s/fold2: running...
   mAP 0.821 (PC 0.821) | - FPS | - W
cbam_yolov5s/fold3: running...
   mAP 0.882 (PC 0.882) | - FPS | - W
cbam_yolov5s/fold4: running...
   mAP 0.865 (PC 0.865) | - FPS | - W
TFLite INT8: evaluating every held-out subject...
   mAP 0.809 (PC 0.805) | - FPS | - W
   mAP 0.402 (PC 0.401) | 

## 5. CPU speed and memory (1 and 4 threads)

In [13]:
# ---- CPU speed and memory of the TFLite models on this PC (GPU not used) ----
# 1 thread and 4 threads stand in for small edge CPUs. Speed does not depend on the fold, so one fold is timed.
# Each model runs in its own process; frames are already in memory (like a camera stream), batch 1.
# Every run is repeated each time this cell runs, so all timings come from one session under the same conditions:
# close other programs (and do not train models) while it runs.
SPEED_FOLD     = FOLDS[0]
SPEED_THREADS  = [1, 4]
SPEED_VARIANTS = ['fp32', 'fp16', 'int8', 'int8m']
SPEED_SECONDS  = 20            # length of each timed run
cores = os.cpu_count() or 1
if any(t > cores for t in SPEED_THREADS):
    print(f'This PC has {cores} logical cores - skipping thread counts above that.')
    SPEED_THREADS = [t for t in SPEED_THREADS if t <= cores]
for t in SPEED_THREADS:
    for v in SPEED_VARIANTS:
        print(f'--- {VARIANT_LABEL[v]}, {t} thread(s)', flush=True)
        run_live([EDGE_PYTHON, bench, '--backend', 'tflite', '--variant', v, '--threads', t, '--skip_accuracy',
                  '--folds', SPEED_FOLD, '--seconds', SPEED_SECONDS, '--redo'],
                 os.path.join(TFLITE_DIR, f'speed_{v}_t{t}.log'), keep=lambda l: l.rstrip().endswith('running...') or l.startswith('   ') or '!!!' in l)


--- FP32, 1 thread(s)
ssd/fold1: running...
   mAP - (PC 0.805) | 14.5 FPS | - W
cbam_ssd/fold1: running...
   mAP - (PC 0.803) | 14.3 FPS | - W
yolov5s/fold1: running...
   mAP - (PC 0.965) | 6.9 FPS | - W
cbam_yolov5s/fold1: running...
   mAP - (PC 0.934) | 4.5 FPS | - W
--- FP16, 1 thread(s)
ssd/fold1: running...
   mAP - (PC 0.805) | 13.2 FPS | - W
cbam_ssd/fold1: running...
   mAP - (PC 0.803) | 12.7 FPS | - W
yolov5s/fold1: running...
   mAP - (PC 0.965) | 6.0 FPS | - W
cbam_yolov5s/fold1: running...
   mAP - (PC 0.934) | 5.8 FPS | - W
--- INT8, 1 thread(s)
ssd/fold1: running...
   mAP - (PC 0.805) | 15.3 FPS | - W
cbam_ssd/fold1: running...
   mAP - (PC 0.803) | 14.1 FPS | - W
yolov5s/fold1: running...
   mAP - (PC 0.965) | 8.5 FPS | - W
cbam_yolov5s/fold1: running...
   mAP - (PC 0.934) | 1.5 FPS | - W
--- INT8, CBAM in float, 1 thread(s)
cbam_ssd/fold1: running...
   mAP - (PC 0.803) | 13.9 FPS | - W
--- FP32, 4 thread(s)
ssd/fold1: running...
   mAP - (PC 0.805) | 21.5 FPS | 

## 6. Results table for the paper

In [14]:
# ---- Complexity + final tables (saved to runs/loso/edge_pc_results.md) ----
COMPLEXITY = {}
for model in MODELS:
    onnx_path = os.path.join(EDGE_EXPORT, model, FOLDS[0], 'model.onnx')
    if os.path.exists(onnx_path):
        r = subprocess.run([EDGE_PYTHON, os.path.join(SCRIPTS_DIR, 'edge_pc.py'), 'complexity', '--onnx', onnx_path],
                           capture_output=True, text=True, env=child_env())
        try:
            COMPLEXITY[model] = json.loads(r.stdout.strip().splitlines()[-1])
        except (ValueError, IndexError):
            print(f'complexity failed for {model}:', r.stderr[-500:])

speed = {}
for fn in os.listdir(rows_dir):
    with open(os.path.join(rows_dir, fn), encoding='utf-8') as f:
        r = json.load(f)
    if 'fps' in r:
        speed[(r['model'], r['precision'], r['threads'])] = r
sizes = {}
for model in MODELS:
    for v in TFLITE_VARIANTS:
        p_ = os.path.join(TFLITE_DIR, 'models', model, FOLDS[0], f'model_{v}.tflite')
        if os.path.exists(p_):
            sizes[(model, v)] = os.path.getsize(p_) / 1e6
device, dev_path = {}, os.path.join(TFLITE_DIR, 'results_tflite', 'edge_results.json')
if os.path.exists(dev_path):
    with open(dev_path, encoding='utf-8') as f:
        device = json.load(f).get('device', {})

L = ['# Edge-oriented benchmark on a PC CPU (TensorFlow Lite)', '',
     f"- CPU: {device.get('cpu', '?')} ({device.get('physical_cores', '?')} cores / {device.get('logical_cores', '?')} threads), "
     f"{device.get('ram_gb', '?')} GB RAM, {device.get('os', '')}; GPU not used",
     '- TensorFlow Lite (XNNPACK CPU kernels), batch 1, frames already in memory. INT8 = full-integer post-training '
     'quantization calibrated on 200 training-subject images per fold.',
     '- This PC stands in for an edge device: the accuracy results hold on any hardware, the speeds do not.', '',
     '## Model size and compute', '',
     '| Model | Parameters | GFLOPs (input) | FP32 MB | FP16 MB | INT8 MB | INT8, CBAM in float MB |', '|---|---|---|---|---|---|---|']
for model in MODELS:
    c = COMPLEXITY.get(model, {})
    inp = speed.get((model, 'fp32', SPEED_THREADS[0]), {}).get('input') or ''
    L.append(f"| {LABELS[model]} | {c.get('params', 0) / 1e6:.2f} M | {c.get('gflops', 0):.2f} ({'x'.join(map(str, inp[::-1])) if inp else '-'}) | "
             + ' | '.join(f"{sizes[(model, v)]:.1f}" if (model, v) in sizes else '-' for v in TFLITE_VARIANTS) + ' |')
L += ['', f'## Accuracy on the held-out subjects (mean ± std over {len(FOLDS)} LOSO folds)', '',
      '| Model | mAP@0.5 Phase B | mAP@0.5 TFLite FP32 | mAP@0.5 TFLite INT8 | INT8 − FP32 | Precision INT8 | Recall INT8 | mAP@0.5 INT8, CBAM in float |',
      '|---|---|---|---|---|---|---|---|']
fmt = lambda t, d=3: f'{t[0]:.{d}f} ± {t[1]:.{d}f}' if t and t[0] is not None else '-'
for model, a in ACCURACY.items():
    d = a['int8'][0] - a['fp32'][0] if a['int8'][0] is not None else None
    L.append(f"| {LABELS[model]} | {fmt(a['phaseB'])} | {fmt(a['fp32'])} | {fmt(a['int8'])} | "
             f"{(f'{d:+.3f}' if d is not None else '-')} | {fmt(a['p_int8'])} | {fmt(a['r_int8'])} | {fmt(a.get('int8m'))} |")
L += ['', f'## CPU speed ({SPEED_FOLD} models; confidence 0.5)', '',
      '| Model | Format | Threads | Pre ms | Inference ms | Post ms | End-to-end ms (p95) | FPS | Model memory MB | Process peak MB |',
      '|---|---|---|---|---|---|---|---|---|---|']
for model in MODELS:
    for v in SPEED_VARIANTS:
        for t in SPEED_THREADS:
            r = speed.get((model, v, t))
            if not r:
                continue
            tm = r['timing_ms']
            L.append(f"| {LABELS[model]} | {VARIANT_LABEL[v]} | {t} | {tm['pre']['mean']:.1f} | {tm['infer']['mean']:.1f} | "
                     f"{tm['post']['mean']:.1f} | {tm['total']['mean']:.1f} ({tm['total']['p95']:.1f}) | {r['fps']:.1f} | "
                     f"{(r.get('model_memory_mb') or 0):.0f} | {(r.get('process_rss_mb') or 0):.0f} |")
L += ['', 'Pre = resize/letterbox; Post = box decoding + NMS (numpy). Model memory = memory the model adds while running '
      '(weights, activations, buffers), measured after the runtime library is loaded. Process peak = whole benchmark '
      'process including the TensorFlow library used here as the TFLite runtime (an edge device uses the much smaller '
      'TFLite runtime library).']
text = '\n'.join(L) + '\n'
out_md = os.path.join(RESULTS_ROOT, 'edge_pc_results.md')
with open(out_md, 'w', encoding='utf-8') as f:
    f.write(text)
print(text)
print('Saved:', out_md)


# Edge-oriented benchmark on a PC CPU (TensorFlow Lite)

- CPU: Intel(R) Core(TM) i7-7700HQ CPU @ 2.80GHz (4 cores / 8 threads), 23.8 GB RAM, Windows-10-10.0.19045-SP0; GPU not used
- TensorFlow Lite (XNNPACK CPU kernels), batch 1, frames already in memory. INT8 = full-integer post-training quantization calibrated on 200 training-subject images per fold.
- This PC stands in for an edge device: the accuracy results hold on any hardware, the speeds do not.

## Model size and compute

| Model | Parameters | GFLOPs (input) | FP32 MB | FP16 MB | INT8 MB | INT8, CBAM in float MB |
|---|---|---|---|---|---|---|
| SSD MobileNetV2 FPNLite 320 | 2.55 M | 1.70 (320x320) | 11.3 | 5.7 | 3.3 | - |
| CBAM-SSD (proposed) | 2.56 M | 1.71 (320x320) | 11.4 | 5.8 | 3.3 | 3.6 |
| YOLOv5s 416 | 7.03 M | 5.14 (416x320) | 28.2 | 14.1 | 7.4 | - |
| CBAM-YOLOv5s (proposed) | 7.08 M | 5.14 (416x320) | 28.4 | 14.2 | 7.4 | - |

## Accuracy on the held-out subjects (mean ± std over 4 LOSO folds)

| Model | mAP@0.5 